<a href="https://colab.research.google.com/github/jeffheaton/app_generative_ai/blob/main/t81_559_class_11_4_mcp_multi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-559: Applications of Generative Artificial Intelligence
**Module 11: Model Context Protocol (MCP)**
* Instructor: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.wustl.edu/Programs/Pages/default.aspx)
* For more information visit the [class website](https://github.com/jeffheaton/app_generative_ai).

# Module 11 Material

* Part 11.1: Introduction to the Model Context Protocol [[Video]]() [[Notebook]](t81_559_class_11_1_mcp.ipynb)
* Part 11.2: Using MCP Servers from an Agent [[Video]]() [[Notebook]](t81_559_class_11_2_mcp_client.ipynb)
* Part 11.3: Building Your Own MCP Server [[Video]]() [[Notebook]](t81_559_class_11_3_mcp_server.ipynb)
* **Part 11.4: MCP Resources and Multi-Server Agents** [[Video]]() [[Notebook]](t81_559_class_11_4_mcp_multi.ipynb)
* Part 11.5: MCP Security and the Road Ahead [[Video]]() [[Notebook]](t81_559_class_11_5_mcp_security.ipynb)

# Google CoLab Instructions

The following code ensures that Google CoLab is running and maps Google Drive if needed.

In [ ]:
import os

try:
    from google.colab import drive, userdata
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# OpenAI Secrets
if COLAB:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Install needed libraries in CoLab
if COLAB:
    !pip install langchain langchain_openai langchain-mcp-adapters "mcp<2" mcp-server-time

# Part 11.4: MCP Resources and Multi-Server Agents

Tools are actions the *model* decides to take. MCP's second primitive, **resources**, is different: a resource is data -- identified by a URI -- that the *application* chooses to read into context. Think of tools as verbs and resources as documents. A server for your company's wiki might expose a `search` tool for the model, and also expose each page as a resource such as `wiki://onboarding` so the application can load one directly, no model call required.

Our data will be familiar: the five hundred fictional employee biographies used in Modules 5 and 7. The server below exposes each company's biographies as a resource (`bios://DD`, `bios://FT`, and so on) and offers one tool, `search_bios`, that finds people by name.

In [ ]:
%%writefile bios_server.py
from mcp.server.fastmcp import FastMCP
import requests

mcp = FastMCP("Bios")

BASE = "https://data.heatonresearch.com/data/t81-559/bios"
COMPANIES = ["DD", "FT", "GS", "NGS", "TI"]
_cache = {}

def _load(company: str):
    if company not in _cache:
        text = requests.get(f"{BASE}/{company}.txt").text
        _cache[company] = [b.strip() for b in text.split("\n\n") if b.strip()]
    return _cache[company]

@mcp.resource("bios://{company}")
def company_bios(company: str) -> str:
    """All employee biographies for one company code (DD, FT, GS, NGS, or TI)."""
    return "\n\n".join(_load(company))

@mcp.tool()
def search_bios(name: str) -> str:
    """Return biography paragraphs that mention the given person's name."""
    hits = []
    for company in COMPANIES:
        hits.extend(b for b in _load(company) if name.lower() in b.lower())
    return "\n\n".join(hits[:3]) if hits else "No biography found for that name."

if __name__ == "__main__":
    mcp.run(transport="stdio")

## Reading a Resource

The application reads resources through the client session -- deliberately *without* involving the language model. The following opens a session to the bios server and loads one company's biographies directly.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.resources import load_mcp_resources

client = MultiServerMCPClient({
    "bios": {
        "transport": "stdio",
        "command": "python",
        "args": ["bios_server.py"],
    },
})

async with client.session("bios") as session:
    resources = await load_mcp_resources(session, uris=["bios://DD"])

text = resources[0].as_string()
print(f"Loaded {len(text):,} characters of biographies from bios://DD")
print()
print(text[:400], "...")

No tokens were spent and no model was involved: the application simply pulled data through the protocol. In a production host, resources are what populate context windows -- the user picks a document, the app loads the resource, and *then* the model reasons over it.

## One Agent, Several Servers

The "Multi" in MultiServerMCPClient is the payoff of the whole protocol. The configuration below wires the agent to two completely independent servers -- our bios server and the third-party time server from Part 11.2 -- and asks one question that requires both. Watch the trace route each tool call to the right server.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

MODEL = 'gpt-5.6-luna'

llm = ChatOpenAI(
        model=MODEL,
        temperature=0.2,
        n=1,
        use_responses_api=True  # tool calling on gpt-5.6 models requires the Responses API
    )

client = MultiServerMCPClient({
    "bios": {
        "transport": "stdio",
        "command": "python",
        "args": ["bios_server.py"],
    },
    "time": {
        "transport": "stdio",
        "command": "python",
        "args": ["-m", "mcp_server_time"],
    },
})

tools = await client.get_tools()
print("Agent's combined toolbox:", [t.name for t in tools])
print()

agent = create_agent(llm, tools)

async for step in agent.astream(
    {"messages": [{"role": "user", "content":
        "Which company does Samantha Doyle work for, and what is the current "
        "time in St. Louis? Answer both questions."}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

One agent, one toolbox, two independent processes behind it -- one of which we wrote, one of which we merely installed. Adding a third capability would be four more lines of configuration. This is the N + M world MCP promised in Part 11.1, running in your notebook.

A capability this powerful deserves suspicion, however. Every server you connect is code you are trusting, and every tool result is text you are feeding to your model. The final part of this module is about exactly that.